In [ ]:
import math  # for mathematical operations
from copy import deepcopy  # for copying object

import numpy as np  # for numerical operations
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import zipfile

%matplotlib inline

from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split  # for splitting the dataset into training and testing sets
from sklearn.preprocessing import StandardScaler  # for scaling the data
from sklearn.preprocessing import MinMaxScaler  # for scaling the data
from sklearn.preprocessing import LabelEncoder  # for encoding the labels
from sklearn.preprocessing import OneHotEncoder  # for one-hot encoding the categorical variables

from sklearn.linear_model import LinearRegression  # for linear regression model
from sklearn.linear_model import LogisticRegression  # for logistic regression model
from sklearn.metrics import confusion_matrix  # for calculating confusion matrix for classification task

from sklearn.metrics import mean_squared_error  # for calculating mean squared error
from sklearn.metrics import mean_absolute_error  # for calculating mean absolute error
from sklearn.metrics import r2_score  # for calculating r2 score
from sklearn.metrics import accuracy_score  # for calculating accuracy score
from sklearn.metrics import precision_score  # for calculating precision score
from sklearn.metrics import recall_score  # for calculating recall score
from sklearn.metrics import f1_score  # for calculating f1 score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import SGD



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

#Plot distribution of the stats (e.g., 'Delivery_Time')

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'], inplace=True)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df.dropna(subset=['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs'], inplace=True)
df['Delivery_Time'].fillna(df['Delivery_Time'].mean(), inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
else:
    print("No Duplicate Samples Found.")

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
  print(f"Encoding column: {col}")
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

df

In [ ]:
# Task 5: Write your code here:
# Standardize features using StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")

In [ ]:
# Retrieve CatBoost feature importances and sort them
RandomForest = model["RandomForest Classifier"]
RandomForest_importance = list(zip(X.columns, RandomForest.feature_importances_))
sorted_RandomForest_importance = sorted(RandomForest_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_RandomForest_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('RandomForest Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: